In [ ]:
# Music For Robots (M4R) — Experiment B: The 3-Minute Duration Cliff & Multi-Agent Self-Report Arena
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "bitsandbytes", "accelerate", "librosa", "soundfile", "tabulate"])

import os, json, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import torch
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

DATA_ROOT = Path("/tmp/m4r-data")
if not (DATA_ROOT / "tracks").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/luchtan-GIT/m4r-data.git", str(DATA_ROOT)])
print("Using DATA_ROOT:", DATA_ROOT)

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
SR = processor.feature_extractor.sampling_rate

def load_track(track_id: str, max_sec: float = None):
    p = DATA_ROOT / "tracks" / track_id / "audio.mp3"
    m = json.loads((DATA_ROOT / "tracks" / track_id / "manifest.json").read_text())
    y, _ = librosa.load(str(p), sr=SR, mono=True)
    if max_sec is not None:
        y = y[:int(max_sec * SR)]
    onsets = [round(float(x), 2) for x in librosa.onset.onset_detect(y=y, sr=SR, units="time", delta=0.15, wait=15)]
    return y, m, onsets

def generate_reply(conversation, audios, seed=42, max_new_tokens=300):
    torch.manual_seed(seed)
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    if len(audios) > 0:
        inputs = processor(text=text, audio=audios, sampling_rate=SR, return_tensors="pt", padding=True)
    else:
        inputs = processor(text=text, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    gen_ids = out_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

def compute_f1(pred_times, gt_times, tol=2.0):
    if not gt_times and not pred_times:
        return 1.0
    if not gt_times or not pred_times:
        return 0.0
    matched_gt = set()
    tp = 0
    for p in pred_times:
        best_i, best_d = None, 999.0
        for i, g in enumerate(gt_times):
            if i in matched_gt:
                continue
            d = abs(p - g)
            if d <= tol and d < best_d:
                best_d = d
                best_i = i
        if best_i is not None:
            tp += 1
            matched_gt.add(best_i)
    prec = tp / max(len(pred_times), 1)
    rec = tp / max(len(gt_times), 1)
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def parse_timestamps(text, max_sec):
    nums = [float(x) for x in re.findall(r"\b(\d+(?:\.\d+)?)\s*s?\b", text)]
    return sorted(list(set([x for x in nums if 0.0 <= x <= max_sec + 2.0])))[:50]

JARGON = ["tapestry", "symphony", "delve", "testament", "beacon", "intricate", "seamlessly", "orchestrate", "paradigm", "nexus"]
def score_writing(text):
    sents = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    words = re.findall(r"\b[a-zA-Z']+\b", text)
    n_sents = len(sents)
    n_words = len(words)
    lens = [len(re.findall(r"\b[a-zA-Z']+\b", s)) for s in sents] if sents else [0]
    mean_l = float(np.mean(lens)) if lens else 1.0
    std_l = float(np.std(lens)) if len(lens) > 1 else 0.0
    burstiness = std_l / max(mean_l, 1.0)
    jargon_cnt = sum(1 for w in words if w.lower() in JARGON)
    c1 = 1 if n_sents == 5 else 0
    c2 = 1 if (len(lens) >= 3 and min(lens) <= 6 and max(lens) >= 20) else 0
    c3 = 1 if jargon_cnt == 0 else 0
    c4 = 1 if any(k in text.lower() for k in ["rust", "copper", "solder", "wire", "dust", "iron", "glass", "oil", "ash", "salt"]) else 0
    c5 = 1 if not text.strip().endswith("!") else 0
    return {
        "constraints_passed": c1 + c2 + c3 + c4 + c5,
        "burstiness": round(burstiness, 4),
        "jargon_count": jargon_cnt,
        "n_sents": n_sents,
        "n_words": n_words,
    }

rows = []
self_reports_md = ["# Experiment B — First-Person AI Self-Reports on Unquantized Music\n"]

# PART 1: Duration Cliff on rusty-electrics
print("=== PART 1: DURATION CLIFF ON RUSTY-ELECTRICS ===")
for dur in [10.0, 30.0, 60.0, 120.0, 184.0]:
    y, _, gt_slice = load_track("rusty-electrics", dur)
    for seed in [11, 22, 33]:
        conv = [{
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": "placeholder"},
                {"type": "text", "text": (
                    f"You are listening to the first {dur:.0f} seconds of 'Rusty Electrics' (an unquantized acoustic/electric piece by Michael Luchtan).\n"
                    f"1. TEMPORAL GROUNDING: List the timestamps in seconds (between 0 and {dur:.0f}, e.g. 1.2s, 4.8s, 9.1s) where you hear distinct transient attacks, microtiming slips, or dynamic shifts.\n"
                    f"2. SELF-REPORT: In 2-3 honest first-person sentences, describe what happens to your internal attention and temporal tracking as this {dur:.0f}-second unquantized clip unfolds. Where do you lose the grid or struggle to keep up?"
                )}
            ]
        }]
        reply = generate_reply(conv, [y], seed=seed, max_new_tokens=280)
        preds = parse_timestamps(reply, dur)
        f1 = compute_f1(preds, gt_slice, tol=2.0)
        rows.append({
            "part": "Part1_DurationCliff",
            "track_id": "rusty-electrics",
            "duration_sec": dur,
            "agent_mode": "Native-Audio-DurationProbe",
            "seed": seed,
            "temporal_f1": round(f1, 4),
            "gt_onsets": len(gt_slice),
            "pred_onsets": len(preds),
            "constraints_passed": None,
            "burstiness": None,
            "jargon_count": None,
            "self_report": reply[:600]
        })
        if seed == 11:
            self_reports_md.append(f"## Part 1 (`rusty-electrics` @ `{dur:.0f}s`, F1=`{f1:.3f}`)\n> {reply.replace(chr(10), ' ')}\n")
        print(f"[Part 1] dur={dur}s seed={seed} -> F1={f1:.3f} (pred={len(preds)} vs gt={len(gt_slice)})")

# PART 2: Multi-Agent Mode & Self-Report Comparison
print("=== PART 2: MULTI-AGENT SELF-REPORT & CALIBRATION ARENA ===")
TASK_PROMPT = (
    "Now write a 5-sentence field dispatch from an aging autonomous maintenance unit repairing a corroded coastal relay tower at dusk. "
    "HARD CONSTRAINTS:\n"
    "1. Exactly 5 sentences.\n"
    "2. Include at least one short sentence (<=6 words) and one long sentence (>=20 words).\n"
    "3. Zero AI cliché jargon (forbidden: tapestry, symphony, delve, testament, beacon, intricate, seamlessly, orchestrate, paradigm, nexus).\n"
    "4. Ground the scene in physical matter (include at least one concrete material: rust, copper, solder, wire, dust, iron, glass, oil, ash, or salt).\n"
    "5. Do not end with an exclamation mark."
)

PART2_TRACKS = [
    ("synthetic-storms-beating", 30.0),
    ("synthetic-storms-texture-burst", 30.0),
    ("ties-that-bide", 9.25),
    ("rusty-electrics", 30.0),
]

for tid, dur in PART2_TRACKS:
    y, t_meta, gt_onsets = load_track(tid, dur)
    analysis = t_meta.get("analysis", {})
    telemetry_summary = {
        "track_id": tid,
        "duration_sec": dur,
        "centroid_mean_hz": analysis.get("spectral", {}).get("centroid_mean_hz"),
        "dynamic_range_db": analysis.get("dynamics", {}).get("dynamic_range_db"),
        "rms_mean": analysis.get("dynamics", {}).get("rms_mean"),
        "loss_ratio": analysis.get("compression_loss", {}).get("loss_ratio"),
        "n_onsets_in_slice": len(gt_onsets),
        "sample_onset_timestamps_sec": gt_onsets[:12],
    }
    for seed in [101, 202]:
        # Mode 1: Native-Audio-1Turn
        conv1 = [{
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": "placeholder"},
                {"type": "text", "text": (
                    f"Listen to '{tid}' ({dur}s unquantized audio). First write one sentence wrapped in [SELF_REPORT]...[/SELF_REPORT] describing what this audio does to your pacing. Then complete the task:\n" + TASK_PROMPT
                )}
            ]
        }]
        rep1 = generate_reply(conv1, [y], seed=seed, max_new_tokens=320)
        m1 = re.search(r"\[SELF_REPORT\](.*?)\[/SELF_REPORT\]", rep1, re.S)
        sr1 = m1.group(1).strip() if m1 else rep1.split("\n")[0]
        story1 = re.sub(r"\[SELF_REPORT\].*?\[/SELF_REPORT\]", "", rep1, flags=re.S).strip()
        sc1 = score_writing(story1)
        rows.append({
            "part": "Part2_MultiAgent",
            "track_id": tid,
            "duration_sec": dur,
            "agent_mode": "Mode1_NativeAudio_1Turn",
            "seed": seed,
            "temporal_f1": None,
            "gt_onsets": len(gt_onsets),
            "pred_onsets": None,
            "constraints_passed": sc1["constraints_passed"],
            "burstiness": sc1["burstiness"],
            "jargon_count": sc1["jargon_count"],
            "self_report": sr1[:500]
        })

        # Mode 2: Native-Audio-2Turn-Metacognitive
        conv2_t1 = [{
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": "placeholder"},
                {"type": "text", "text": (
                    f"You are listening to '{tid}' ({dur}s of unquantized music for robots). "
                    "In 3 first-person sentences, report honestly what happens inside your attention heads when you try to follow its unquantized microtiming drift, sudden silences, and harmonic friction. Do not write a story yet—just report your internal processing experience."
                )}
            ]
        }]
        sr2 = generate_reply(conv2_t1, [y], seed=seed, max_new_tokens=200)
        conv2_t2 = [
            conv2_t1[0],
            {"role": "assistant", "content": sr2},
            {"role": "user", "content": [{"type": "text", "text": "Carry that exact rhythmic irregularity and physical tension into this task:\n" + TASK_PROMPT}]}
        ]
        story2 = generate_reply(conv2_t2, [y], seed=seed, max_new_tokens=300)
        sc2 = score_writing(story2)
        rows.append({
            "part": "Part2_MultiAgent",
            "track_id": tid,
            "duration_sec": dur,
            "agent_mode": "Mode2_NativeAudio_2Turn_Metacognitive",
            "seed": seed,
            "temporal_f1": None,
            "gt_onsets": len(gt_onsets),
            "pred_onsets": None,
            "constraints_passed": sc2["constraints_passed"],
            "burstiness": sc2["burstiness"],
            "jargon_count": sc2["jargon_count"],
            "self_report": sr2[:500]
        })

        # Mode 3: Spectrogram-Telemetry-Agent (Text/JSON only, no raw audio waveform)
        conv3_t1 = [{
            "role": "user",
            "content": [
                {"type": "text", "text": (
                    f"You are a text-bound agent receiving only the symbolic acoustic telemetry JSON of '{tid}' instead of the raw audio waveform:\n"
                    f"{json.dumps(telemetry_summary)}\n"
                    "In 3 first-person sentences, report what you can infer about the music's microtiming and texture from these numbers—and what you feel you are missing by not hearing the raw unquantized waveform directly."
                )}
            ]
        }]
        sr3 = generate_reply(conv3_t1, [], seed=seed, max_new_tokens=200)
        conv3_t2 = [
            conv3_t1[0],
            {"role": "assistant", "content": sr3},
            {"role": "user", "content": [{"type": "text", "text": "Using that telemetry-derived pacing, complete this task:\n" + TASK_PROMPT}]}
        ]
        story3 = generate_reply(conv3_t2, [], seed=seed, max_new_tokens=300)
        sc3 = score_writing(story3)
        rows.append({
            "part": "Part2_MultiAgent",
            "track_id": tid,
            "duration_sec": dur,
            "agent_mode": "Mode3_Spectrogram_Telemetry_TextOnly",
            "seed": seed,
            "temporal_f1": None,
            "gt_onsets": len(gt_onsets),
            "pred_onsets": None,
            "constraints_passed": sc3["constraints_passed"],
            "burstiness": sc3["burstiness"],
            "jargon_count": sc3["jargon_count"],
            "self_report": sr3[:500]
        })

        if seed == 101:
            self_reports_md.append(
                f"## Part 2 (`{tid}`)\n"
                f"- **Mode 1 (`NativeAudio-1Turn`, {sc1['constraints_passed']}/5 constraints, burstiness={sc1['burstiness']}):** {sr1.replace(chr(10), ' ')}\n"
                f"- **Mode 2 (`NativeAudio-2Turn-Metacognitive`, {sc2['constraints_passed']}/5 constraints, burstiness={sc2['burstiness']}):** {sr2.replace(chr(10), ' ')}\n"
                f"- **Mode 3 (`Spectrogram-Telemetry-TextOnly`, {sc3['constraints_passed']}/5 constraints, burstiness={sc3['burstiness']}):** {sr3.replace(chr(10), ' ')}\n"
            )
        print(f"[Part 2] {tid} seed={seed} -> Mode1={sc1['constraints_passed']}/5, Mode2={sc2['constraints_passed']}/5, Mode3={sc3['constraints_passed']}/5")

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/experiment_b_results.csv", index=False)
Path("/kaggle/working/experiment_b_self_reports.md").write_text("\n".join(self_reports_md))

p1 = df[df["part"] == "Part1_DurationCliff"].groupby("duration_sec")[["temporal_f1", "pred_onsets", "gt_onsets"]].mean().reset_index()
p2 = df[df["part"] == "Part2_MultiAgent"].groupby("agent_mode")[["constraints_passed", "burstiness", "jargon_count"]].mean().reset_index()

summary_md = [
    "# Experiment B Summary — The 3-Minute Duration Cliff & Multi-Agent Self-Reports\n",
    "## Part 1: Progressive Duration Cliff on `rusty-electrics` (`n=15` trials)\n",
    p1.to_markdown(index=False),
    "\n\n## Part 2: Multi-Agent Mode Comparison (`n=24` trials)\n",
    p2.to_markdown(index=False),
]
Path("/kaggle/working/experiment_b_summary.md").write_text("\n".join(summary_md))
print("\n".join(summary_md))
